# 1. Libraries Installation

In [ ]:
# Install all required libraries (run this cell first on Google Colab)
!pip install tensorflow opencv-python numpy matplotlib scikit-learn tqdm pandas pillow

# 2. Import Libraries

In [ ]:
#Core Libraries
import os
import cv2                  # OpenCV for video/frame handling
import numpy as np          # Numerical operations
import random
import matplotlib.pyplot as plt   # For visualization
import pandas as pd          # (Optional) Logging results
from tqdm import tqdm
from glob import glob
from sklearn.metrics import classification_report  # For evaluation

# 3. Mount Google Drive

Mount Google Drive to access the RWF-2000 dataset and save model outputs.

In [ ]:
# Mount Google Drive to access dataset and save outputs.
# After mounting, the project folder should be available at:
#   /content/drive/MyDrive/ViolenceDetection/
#
# Expected project structure inside Drive:
#   ViolenceDetection/
#   ├── dataset/
#   │   ├── violence/        <- RWF-2000 violent clips
#   │   └── non_violence/    <- RWF-2000 non-violent clips
#   ├── processed_frames/    <- (auto-generated by Cell 4)
#   ├── extracted_features/  <- (auto-generated by Cell 5)
#   ├── models/              <- (auto-generated by Cell 6)
#   └── your_videos/         <- place your own test videos here

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 4. Preprocessing and Frame Extraction

Extract frames from each video at fixed intervals (every 10th frame), resize to 224x224, and save as JPG images.

In [ ]:

# 1) Path configuration
def setup_paths():
    """
    Define dataset and output paths dynamically (inside Drive or local).
    Expected dataset structure:
      /ViolenceDetection/dataset/violence/*.mp4
      /ViolenceDetection/dataset/non_violence/*.mp4
    """
    base_path = "./"  # Set this to your project root (e.g., "/content/drive/MyDrive/ViolenceDetection/")

    dataset_path = os.path.join(base_path, "dataset/")
    output_path  = os.path.join(base_path, "processed_frames/")

    # Ensure directories exist
    os.makedirs(os.path.join(dataset_path, "violence"), exist_ok=True)
    os.makedirs(os.path.join(dataset_path, "non_violence"), exist_ok=True)
    os.makedirs(output_path, exist_ok=True)

    print("Dataset Path:", dataset_path)
    print("Output Path :", output_path)
    return dataset_path, output_path

# 2) Frame extraction
FRAME_INTERVAL = 10        # Extract 1 frame every 10 frames
IMG_SIZE = (224, 224)      # Resize for ResNet50 input

def extract_frames_from_video(video_path, output_folder, frame_interval=FRAME_INTERVAL):
    """
    Extract frames from a video at fixed intervals and save as images.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Could not open video: {video_path}")
        return

    frame_count, saved_count = 0, 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % frame_interval == 0:
            # resize -> RGB -> save
            frame = cv2.resize(frame, IMG_SIZE, interpolation=cv2.INTER_AREA)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            os.makedirs(output_folder, exist_ok=True)
            frame_path = os.path.join(output_folder, f"frame_{saved_count:05d}.jpg")
            cv2.imwrite(frame_path, cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
            saved_count += 1

        frame_count += 1

    cap.release()
    print(f"{saved_count} frames extracted from {os.path.basename(video_path)}")

# 3) Dataset loop
def preprocess_dataset(dataset_path, output_path):
    """
    Structure expected:
        dataset/
            ├── violence/
            └── non_violence/
    """
    categories = ["violence", "non_violence"]

    for category in categories:
        category_path = os.path.join(dataset_path, category)
        output_category_path = os.path.join(output_path, category)
        os.makedirs(output_category_path, exist_ok=True)

        # Add more extensions if needed
        videos = [f for f in os.listdir(category_path)
                  if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv', '.m4v'))]

        print(f"\nProcessing '{category}' videos ({len(videos)} files)...")
        if len(videos) == 0:
            print(f"No videos found in {category_path}")
            continue

        for video_file in tqdm(videos):
            video_path = os.path.join(category_path, video_file)
            video_output_path = os.path.join(output_category_path, os.path.splitext(video_file)[0])
            extract_frames_from_video(video_path, video_output_path)

    print("\nPreprocessing complete! Frames saved in:", output_path)

# 4) (Optional) Normalization utility (for later)
def normalize_frame(img):
    """Normalize image pixels to [0,1] range."""
    return img.astype('float32') / 255.0

# 5) Run
if __name__ == "__main__":
    DATASET_PATH, OUTPUT_PATH = setup_paths()
    preprocess_dataset(DATASET_PATH, OUTPUT_PATH)


# 5. Feature Extraction (ResNet50)

Extract 2048-dimensional feature vectors from each frame using a pre-trained ResNet50 (ImageNet). Output: one `.npy` file per video with shape `(num_frames, 2048)`.

In [ ]:
# ====================== ResNet50 FEATURE EXTRACTION ======================
# Input:  processed_frames/<class>/<video>/frame_*.jpg
# Output: extracted_features/<class>/<video>.npy
# Each saved file shape: (num_frames, 2048) — perfect for LSTM input
# ========================================================================

# --- Imports ---
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

# --- Paths ---
BASE_PATH = "./"  # Set this to your project root (e.g., "/content/drive/MyDrive/ViolenceDetection/")

FRAMES_ROOT = os.path.join(BASE_PATH, "processed_frames")     # expects /violence/<video>/frame_*.jpg
FEATS_ROOT  = os.path.join(BASE_PATH, "extracted_features")   # will write /violence/<video>.npy
os.makedirs(os.path.join(FEATS_ROOT, "violence"), exist_ok=True)
os.makedirs(os.path.join(FEATS_ROOT, "non_violence"), exist_ok=True)

# --- Model (ImageNet, no classifier, global average pooling -> 2048-d) ---
print("TF:", tf.__version__, "GPUs:", tf.config.list_physical_devices('GPU'))
resnet = ResNet50(weights="imagenet", include_top=False, pooling="avg")
IMG_SIZE = (224, 224)

def load_batch(img_paths):
    """Load list of images, resize to 224x224 RGB, return preprocessed batch for ResNet50."""
    if not img_paths:
        return None
    batch = []
    for p in img_paths:
        bgr = cv2.imread(p)
        if bgr is None:
            continue
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
        batch.append(img)
    if not batch:
        return None
    batch = np.asarray(batch, dtype=np.float32)
    batch = preprocess_input(batch)  # ResNet50-specific normalization ([-123.68, -116.78, -103.94] etc.)
    return batch

def extract_video_features(frames_folder, batch_size=32):
    """
    frames_folder: path containing frame_*.jpg
    returns: np.array of shape (num_frames, 2048)
    """
    img_paths = sorted(glob(os.path.join(frames_folder, "frame_*.jpg")))
    if len(img_paths) == 0:
        # try png just in case
        img_paths = sorted(glob(os.path.join(frames_folder, "frame_*.png")))
    if len(img_paths) == 0:
        return np.zeros((0, 2048), dtype=np.float32)

    feats_list = []
    for i in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[i:i+batch_size]
        batch = load_batch(batch_paths)
        if batch is None:
            continue
        # GPU if available; silently falls back to CPU
        with tf.device("/GPU:0"):
            batch_feats = resnet.predict(batch, verbose=0)  # (B, 2048)
        feats_list.append(batch_feats)

    if not feats_list:
        return np.zeros((0, 2048), dtype=np.float32)
    return np.vstack(feats_list)

def save_class_features(class_name):
    """
    For class 'violence' or 'non_violence':
      reads processed_frames/<class>/<video_folder>/frame_*.jpg
      writes extracted_features/<class>/<video_folder>.npy
    """
    frames_class_dir = os.path.join(FRAMES_ROOT, class_name)
    out_class_dir    = os.path.join(FEATS_ROOT,  class_name)
    os.makedirs(out_class_dir, exist_ok=True)

    if not os.path.isdir(frames_class_dir):
        print(f"Skipping '{class_name}': missing folder {frames_class_dir}")
        return

    video_folders = sorted([
        d for d in os.listdir(frames_class_dir)
        if os.path.isdir(os.path.join(frames_class_dir, d))
    ])
    print(f"\nClass '{class_name}': {len(video_folders)} video folders")

    for vid in tqdm(video_folders):
        src_folder = os.path.join(frames_class_dir, vid)
        dst_file   = os.path.join(out_class_dir, f"{vid}.npy")

        if os.path.exists(dst_file):   # idempotent
            continue

        feats = extract_video_features(src_folder, batch_size=32)  # (N, 2048)
        np.save(dst_file, feats)

def extract_all():
    for cls in ["violence", "non_violence"]:
        save_class_features(cls)
    print("\nFeature extraction complete →", FEATS_ROOT)
    print("   Each file: (num_frames, 2048)")

# --- Run ---
extract_all()

# --- Quick sanity check ---
v_dir = os.path.join(FEATS_ROOT, "violence")
if os.path.isdir(v_dir):
    npys = sorted([f for f in os.listdir(v_dir) if f.endswith(".npy")])
    if npys:
        sample = os.path.join(v_dir, npys[0])
        arr = np.load(sample, allow_pickle=False)
        print("Sample:", os.path.basename(sample), "shape:", arr.shape)


# 6. Temporal Modeling (LSTM)

Train a Bidirectional LSTM classifier on the extracted ResNet50 features. Each video is represented as a fixed-length sequence of 16 feature vectors (2048-dim each).

In [ ]:
# ====================== TEMPORAL MODELING (LSTM) ======================
# Input:  extracted_features/<class>/<video>.npy
# Each .npy: (num_frames, 2048)
# Output: trained LSTM classifier and evaluation metrics
# =====================================================================

import os, numpy as np, random
from glob import glob
from tqdm import tqdm
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# ---------------- Paths & Params ---------------- #
BASE = "./"  # Set this to your project root (e.g., "/content/drive/MyDrive/ViolenceDetection/")
FEATS_DIR = f"{BASE}/extracted_features"  # expects /violence/*.npy, /non_violence/*.npy

SEQ_LEN = 16           # fixed number of frames per clip (pad/truncate)
FEAT_DIM = 2048        # ResNet50 (pooling='avg') output size
TEST_SIZE = 0.15
VAL_SIZE  = 0.15
BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-3
MODEL_DIR = f"{BASE}/models"
os.makedirs(MODEL_DIR, exist_ok=True)

# ---------------- Utilities ---------------- #
def pad_or_truncate(seq: np.ndarray, max_len=SEQ_LEN):
    """
    seq: (T, 2048)
    If T < max_len -> pad with zeros at end.
    If T > max_len -> uniformly sample max_len indices.
    Returns (max_len, 2048)
    """
    T = seq.shape[0]
    if T == 0:
        # empty -> all zeros
        return np.zeros((max_len, FEAT_DIM), dtype=np.float32)

    if T == max_len:
        return seq.astype(np.float32)

    if T < max_len:
        out = np.zeros((max_len, FEAT_DIM), dtype=np.float32)
        out[:T] = seq.astype(np.float32)
        return out

    # T > max_len: uniform sampling for coverage
    idxs = np.linspace(0, T-1, max_len).astype(int)
    return seq[idxs].astype(np.float32)

def load_dataset(feats_dir=FEATS_DIR, seq_len=SEQ_LEN):
    """
    Scans:
      feats_dir/violence/*.npy  -> label 1
      feats_dir/non_violence/*.npy -> label 0
    Pads/truncates to (seq_len, FEAT_DIM).
    Returns X, y (numpy arrays)
    """
    X, y = [], []

    for cls, label in [("violence", 1), ("non_violence", 0)]:
        folder = os.path.join(feats_dir, cls)
        files = sorted(glob(os.path.join(folder, "*.npy")))
        print(f"{cls}: {len(files)} files")

        for fpath in tqdm(files, desc=f"Loading {cls}"):
            seq = np.load(fpath)  # (T, 2048)
            seq_fixed = pad_or_truncate(seq, max_len=seq_len)
            X.append(seq_fixed)
            y.append(label)

    X = np.stack(X).astype(np.float32)   # (N, seq_len, 2048)
    y = np.array(y, dtype=np.int32)      # (N,)
    print("X:", X.shape, "y:", y.shape, "positives:", y.sum(), "negatives:", len(y)-y.sum())
    return X, y

# ---------------- Load data ---------------- #
X, y = load_dataset()

# Train/Val/Test split
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=42
)
val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio, stratify=y_temp, random_state=42
)
print("Splits ->",
      "train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

# ---------------- Build LSTM model ---------------- #
def build_lstm_model(seq_len=SEQ_LEN, feat_dim=FEAT_DIM):
    model = Sequential([
        # Optional masking if you keep true padding zeros (we're padding manually anyway)
        Masking(mask_value=0.0, input_shape=(seq_len, feat_dim)),
        Bidirectional(LSTM(128, return_sequences=False)),
        Dropout(0.5),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')  # binary classification
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_lstm_model()
model.summary()

# ---------------- Train ---------------- #
ckpt_path = os.path.join(MODEL_DIR, "best_lstm.keras")
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# ---------------- Plot training curves ---------------- #
plt.figure()
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure()
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# ---------------- Evaluate ---------------- #
y_prob = model.predict(X_test, batch_size=BATCH_SIZE).ravel()
y_pred = (y_prob >= 0.5).astype(np.int32)

acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)
print(f"\nTest Accuracy: {acc:.4f}")
print(f"Precision:     {prec:.4f}")
print(f"Recall:        {rec:.4f}")
print(f"F1-score:      {f1:.4f}")

print("\nClassification report:\n", classification_report(y_test, y_pred, target_names=["non_violence","violence"]))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# ---------------- Save final model ---------------- #
final_path = os.path.join(MODEL_DIR, "lstm_final.keras")
model.save(final_path)
print("\nSaved best to:", ckpt_path)
print("Saved final to:", final_path)

# ---------------- Inference helper ---------------- #
def predict_clip(feature_file, seq_len=SEQ_LEN, threshold=0.5):
    """
    feature_file: path to a single video's .npy (T, 2048)
    returns: (prob_violent, label_str)
    """
    seq = np.load(feature_file)
    seq_fixed = pad_or_truncate(seq, max_len=seq_len)[None, ...]  # (1, seq_len, 2048)
    prob = float(model.predict(seq_fixed, verbose=0)[0,0])
    return prob, ("violent" if prob >= threshold else "non_violent")

# Example (optional): run on one sample file if available
sample_dir = os.path.join(FEATS_DIR, "violence")
if os.path.isdir(sample_dir):
    npys = sorted(glob(os.path.join(sample_dir, "*.npy")))
    if npys:
        p, lbl = predict_clip(npys[0])
        print(f"\nSample: {os.path.basename(npys[0])} -> P(violent)={p:.3f} => {lbl}")


# 7. Classification & Visualization

Run batch classification on your own videos. For each video: sample 16 frames, extract ResNet50 features, classify with LSTM, and display a probability chart with a representative snapshot.

In [ ]:
# ====================== BATCH CLASSIFICATION + VISUALIZATION ======================
# Scans your_videos/*.(mp4/mov/avi/mkv/m4v/wmv/flv)
# For each video:
#   - Reads frames, uniformly samples SEQ_LEN frames
#   - Extracts ResNet50 features (2048-d per frame)
#   - Runs LSTM classifier (best_lstm.keras)
#   - Prints Violent/Non-Violent + bar chart + a representative snapshot
# Saves overall results to: your_videos_results.txt
# ================================================================================

# --- (Optional) if some formats fail to read, uncomment to install ffmpeg ---
# !apt -y install ffmpeg

# --- Imports ---
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import load_model

# --- Paths & Params ---
BASE = "./"  # Set this to your project root (e.g., "/content/drive/MyDrive/ViolenceDetection/")
MODEL_PATH  = f"{BASE}/models/best_lstm.keras"
YOUR_VIDEOS = f"{BASE}/your_videos"
REPORT_PATH = f"{BASE}/your_videos_results.txt"

SEQ_LEN   = 16         # must match your training
FEAT_DIM  = 2048       # ResNet50 (pooling='avg') output size
THRESHOLD = 0.50       # tune per your tolerance for false positives
IMG_SIZE  = (224, 224)

# Ensure folder exists
os.makedirs(YOUR_VIDEOS, exist_ok=True)

# --- Load models ---
print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices('GPU'))
# Feature extractor (ImageNet weights, no top, global average pooling -> 2048)
resnet = ResNet50(weights="imagenet", include_top=False, pooling="avg")
# LSTM classifier (your trained model)
lstm_model = load_model(MODEL_PATH)
print("Models loaded from:", MODEL_PATH)

# --- Helpers: video I/O, feature extraction, classification ---
def read_video_all_frames(video_path):
    """Read all frames (BGR) from a video file into a list."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    frames = []
    while True:
        ret, f = cap.read()
        if not ret:
            break
        frames.append(f)  # BGR
    cap.release()
    return frames

def uniform_sample_indices(total, num_samples):
    """Uniformly sample exactly num_samples indices from [0..total-1]."""
    if total <= 0:
        return np.array([], dtype=int)
    return np.linspace(0, total-1, num_samples).astype(int)

def preprocess_frame_bgr_to_rgb_resized(bgr):
    """Resize to IMG_SIZE and convert BGR->RGB, float32."""
    img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    return img.astype(np.float32)

def frames_to_resnet_features(frames_rgb, batch_size=32):
    """
    frames_rgb: list/array of RGB frames (H,W,3), float32 in [0..255] (preprocess_input handles scaling)
    returns: (N, 2048)
    """
    if len(frames_rgb) == 0:
        return np.zeros((0, FEAT_DIM), dtype=np.float32)

    feats = []
    for i in range(0, len(frames_rgb), batch_size):
        batch = np.stack(frames_rgb[i:i+batch_size], axis=0)  # (B, 224, 224, 3)
        batch = preprocess_input(batch)                       # ResNet50-specific normalization
        try:
            with tf.device("/GPU:0"):
                f = resnet.predict(batch, verbose=0)
        except:
            f = resnet.predict(batch, verbose=0)
        feats.append(f)
    return np.vstack(feats).astype(np.float32)

def classify_video_fixedlen(video_path, seq_len=SEQ_LEN, threshold=THRESHOLD):
    """
    Read full video, uniformly sample seq_len frames, extract features, predict with LSTM.
    Returns: prob (float), label ("violent"|"non_violent"), sampled_frames_rgb (np.array of sampled frames)
    """
    bgr_frames = read_video_all_frames(video_path)
    if len(bgr_frames) == 0:
        raise ValueError("Video has no frames or cannot be read.")

    idxs = uniform_sample_indices(len(bgr_frames), seq_len)
    rgb_frames = [preprocess_frame_bgr_to_rgb_resized(bgr_frames[i]) for i in idxs]  # list of RGB frames
    feats = frames_to_resnet_features(rgb_frames)             # (seq_len, 2048)
    feats = feats[None, ...]                                  # (1, seq_len, 2048)
    prob = float(lstm_model.predict(feats, verbose=0)[0, 0])
    label = "violent" if prob >= threshold else "non_violent"
    sampled_frames_rgb = np.stack(rgb_frames, axis=0)         # (seq_len, 224, 224, 3)
    return prob, label, sampled_frames_rgb

def show_result_charts(video_filename, prob, label, sampled_frames_rgb):
    """
    Shows: probability bar chart + a representative snapshot (middle frame) with label.
    """
    # 1) probability bar chart
    plt.figure(figsize=(5,3))
    plt.bar(["Non-Violent", "Violent"], [1.0 - prob, prob])
    plt.ylim(0, 1)
    plt.ylabel("Probability")
    plt.title(f"{video_filename}\nP(Violence)={prob:.3f} → {label.upper()}")
    plt.text(1, min(0.98, prob + 0.02), f"{prob:.2f}", ha="center")
    plt.show()

    # 2) representative frame (middle of sampled sequence)
    mid = len(sampled_frames_rgb) // 2
    frame_mid = sampled_frames_rgb[mid].astype(np.uint8)
    plt.figure()
    plt.imshow(frame_mid)
    plt.title(f"{label.upper()} (P={prob:.3f})")
    plt.axis("off")
    plt.show()

# --- Collect videos (multiple formats) ---
EXTS = (".mp4", ".mov", ".avi", ".mkv", ".m4v", ".wmv", ".flv")
videos = sorted([p for p in glob(os.path.join(YOUR_VIDEOS, "*")) if p.lower().endswith(EXTS)])
print(f"\nFound {len(videos)} video(s) in: {YOUR_VIDEOS}")

# --- Batch classify & visualize ---
lines = ["video\tprob_violent\tlabel"]
for vpath in tqdm(videos):
    vname = os.path.basename(vpath)
    try:
        prob, label, sampled = classify_video_fixedlen(vpath, SEQ_LEN, THRESHOLD)
        print(f"\n{vname}: P(violence)={prob:.4f} → {label.upper()}")
        lines.append(f"{vname}\t{prob:.4f}\t{label}")
        # Show chart and snapshot
        show_result_charts(vname, prob, label, sampled)
    except Exception as e:
        print(f"{vname}: ERROR -> {e}")
        lines.append(f"{vname}\tERROR\t{e}")

# --- Save report ---
with open(REPORT_PATH, "w") as f:
    f.write("\n".join(lines))
print(f"\nFinished. Report saved to: {REPORT_PATH}")
